# Deep Learning Project - Sales Demand Prediction using ANN
**Topics:** ANN, Forward Pass, Backpropagation, Adam Optimizer, RMSProp, Sigmoid, Tanh, ReLU, Naive Bayes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('sales_data.csv')
print('Shape:', df.shape)
print('Missing:', df.isnull().sum().sum())
df.head()

In [ ]:
df.describe().T

## 2. Feature Engineering & Preprocessing

In [ ]:
features = ['Inventory Level', 'Units Sold', 'Units Ordered', 'Price',
            'Discount', 'Promotion', 'Competitor Pricing', 'Epidemic']

for col, name in [('Category','Category_enc'), ('Region','Region_enc'),
                   ('Weather Condition','Weather_enc'), ('Seasonality','Season_enc')]:
    df[name] = LabelEncoder().fit_transform(df[col])

all_features = features + ['Category_enc', 'Region_enc', 'Weather_enc', 'Season_enc']
X = df[all_features].values
y = df['Demand'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

X_tr = torch.FloatTensor(X_train_s)
y_tr = torch.FloatTensor(y_train).unsqueeze(1)
X_te = torch.FloatTensor(X_test_s)
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=256, shuffle=True)

print(f'Train: {len(X_train)}, Test: {len(X_test)}, Features: {X_train.shape[1]}')

## 3. ANN Model (Forward Pass + Backpropagation)

In [ ]:
class DemandANN(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):         # Forward Pass
        return self.net(x)

def train(model, optimizer, loader, epochs=50):
    criterion = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        total = 0
        for xb, yb in loader:
            pred = model(xb)            # Forward pass
            loss = criterion(pred, yb)
            optimizer.zero_grad()        # Clear gradients
            loss.backward()              # Backpropagation
            optimizer.step()             # Update weights
            total += loss.item()
        losses.append(total / len(loader))
        if (ep+1) % 10 == 0:
            print(f'  Epoch {ep+1}/{epochs} | Loss: {losses[-1]:.4f}')
    return losses

n = X_train.shape[1]

## 4. Activation Functions (Sigmoid, Tanh, ReLU)

In [ ]:
x = torch.linspace(-5, 5, 100)
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, d, t in zip(ax, [torch.sigmoid(x), torch.tanh(x), torch.relu(x)],
                       ['Sigmoid', 'Tanh', 'ReLU']):
    a.plot(x.numpy(), d.numpy(), lw=2); a.set_title(t)
    a.grid(True, alpha=0.3); a.axhline(0,color='k',lw=0.5); a.axvline(0,color='k',lw=0.5)
plt.tight_layout(); plt.show()

## 5. Train with RMSProp & Adam Optimizers

In [ ]:
print('=== RMSProp ===')
m1 = DemandANN(n)
l1 = train(m1, torch.optim.RMSprop(m1.parameters(), lr=0.001), loader)

print('\n=== Adam ===')
m2 = DemandANN(n)
l2 = train(m2, torch.optim.Adam(m2.parameters(), lr=0.001), loader)

plt.figure(figsize=(8, 4))
plt.plot(l1, label='RMSProp', lw=2)
plt.plot(l2, label='Adam', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('RMSProp vs Adam'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 6. Evaluate Model

In [ ]:
m2.eval()
with torch.no_grad():
    y_pred = m2(X_te).numpy().flatten()

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R2:   {r2:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.4f}')
print(f'\nML project R2 was 0.7405 -- DL ANN R2 is {r2:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_test[:500], y_pred[:500], alpha=0.4, s=10)
ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax[0].set_xlabel('Actual'); ax[0].set_ylabel('Predicted')
ax[0].set_title(f'Actual vs Predicted (R2={r2:.4f})'); ax[0].grid(True, alpha=0.3)
ax[1].hist(y_test - y_pred, bins=50, color='green', alpha=0.7, edgecolor='black')
ax[1].set_title('Residuals'); ax[1].axvline(0, color='r', ls='--'); ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Naive Bayes (Classification)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

def label(d): return 'Low' if d < 70 else ('Medium' if d < 130 else 'High')

yc_tr = np.array([label(d) for d in y_train])
yc_te = np.array([label(d) for d in y_test])

nb = GaussianNB()
nb.fit(X_train_s, yc_tr)
nb_pred = nb.predict(X_test_s)

print(f'Accuracy: {accuracy_score(yc_te, nb_pred):.4f}')
print(classification_report(yc_te, nb_pred))

## 8. Save Model

In [ ]:
torch.save({
    'model_state': m2.state_dict(),
    'input_dim': n,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'r2_score': r2,
    'rmse': rmse
}, 'demand_ann_model.pth')
print('Saved demand_ann_model.pth')

---
## 9. Flask Web App
Run this cell then open **http://127.0.0.1:5000**

In [ ]:
from flask import Flask, request, jsonify, send_file
import threading, torch, numpy as np, torch.nn as nn

# Load saved model
class DemandANN(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1))
    def forward(self, x): return self.net(x)

ckpt = torch.load('demand_ann_model.pth', map_location='cpu', weights_only=False)
model = DemandANN(ckpt['input_dim'])
model.load_state_dict(ckpt['model_state'])
model.eval()
s_mean = np.array(ckpt['scaler_mean'])
s_scale = np.array(ckpt['scaler_scale'])

# Flask app
app = Flask(__name__)

@app.route('/')
def home():
    return send_file('index.html')

@app.route('/api/predict', methods=['POST'])
def predict():
    d = request.json
    f = np.array([[float(d['inventory_level']), float(d['units_sold']),
                   float(d['units_ordered']), float(d['price']),
                   float(d['discount']), int(d['promotion']),
                   float(d['competitor_pricing']), int(d['epidemic']),
                   int(d.get('category',0)), int(d.get('region',0)),
                   int(d.get('weather',0)), int(d.get('season',0))]])
    f = (f - s_mean) / s_scale
    with torch.no_grad():
        pred = model(torch.FloatTensor(f)).item()
    return jsonify({'predicted_demand': round(pred, 2),
                    'model_r2': round(ckpt['r2_score'], 4),
                    'model_rmse': round(ckpt['rmse'], 2)})

# Run Flask in background thread
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()
print('Server running! Open: http://127.0.0.1:5000')